In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
!pip install -r requirements.txt

In [0]:
from src import *

In [0]:
gtfs_df = spark.read.parquet("/Volumes/workspace/default/bmtceta/raw_gtfs/routes_sequential.parquet")

stop_df = spark.read.csv("/Volumes/workspace/default/bmtceta/raw_gtfs/routes_sequential.csv", header = True)

In [0]:
stop_df = standardize_stop_sequence(stop_df)
gtfs_df = standardize_gtfs(gtfs_df)

In [0]:
snapped_df = snap_gtfs(gtfs_df, stop_df)

In [0]:
arrival_df = extract_arrivals(snapped_df)


In [0]:
travel_df = calculate_travel_times(arrival_df)

In [0]:
historical_df = build_historical_table(travel_df)

In [0]:
features_df = build_feature_table(travel_df, historical_df)

In [0]:
feature_df = build_feature_table(
    travel_df,
    historical_df
)

validate_features(feature_df)

train_df, test_df = prepare_train_test(
    feature_df
)

model, assembler = train_random_forest(
    train_df
)

prediction_df = predict(
    model,
    assembler,
    test_df
)

metrics = evaluate(
    prediction_df
)

print_metrics(metrics)